### Data Platforms with Bruin

In [26]:
import duckdb
import pandas as pd

**Note:** Jupyter kernel is running inside of the following directory, not the notebook's directory:

`zoomcamp/data-engineering-zoomcamp/04-analytics-engineering/taxi_rides_ny`

In [27]:
# Connect to the db.
con = duckdb.connect("taxi_rides_ny.duckdb")

In [28]:
#con.sql("DROP TABLE IF EXISTS staging.trips")
#con.sql("DROP TABLE IF EXISTS reports.trips_report")

In [29]:
# Check all schemata.
con.execute("SELECT * FROM information_schema.schemata").df()

,catalog_name,schema_name,schema_owner,default_character_set_catalog,default_character_set_schema,default_character_set_name,sql_path
0,system,information_schema,duckdb,None,None,None,None
1,system,main,duckdb,None,None,None,None
2,system,pg_catalog,duckdb,None,None,None,None
3,taxi_rides_ny,dev,duckdb,None,None,None,None
4,taxi_rides_ny,ingestion,duckdb,None,None,None,None
5,taxi_rides_ny,ingestion_staging,duckdb,None,None,None,None
6,taxi_rides_ny,main,duckdb,None,None,None,None
7,taxi_rides_ny,prod,duckdb,None,None,None,None
8,taxi_rides_ny,reports,duckdb,None,None,None,None
9,taxi_rides_ny,staging,duckdb,None,None,None,None


In [30]:
con.sql("SELECT * FROM staging.trips LIMIT 5").df()

,vendor_id,rate_code_id,pickup_location_id,dropoff_location_id,pickup_datetime,dropoff_datetime,store_and_fwd_flag,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,payment_type_description,service_type,extracted_at
0,2.0,1.0,42,244,2021-01-02 00:14:34,2021-01-02 00:19:53,N,1.0,1.25,6.0,0.50,0.5,0.00,0.0,0.3,7.30,1.0,credit_card,green,2026-02-22 14:36:58.044440-05:00
1,2.0,1.0,75,220,2021-01-02 00:21:20,2021-01-02 00:52:01,N,1.0,9.11,31.0,0.50,0.5,0.00,0.0,0.3,32.30,1.0,credit_card,green,2026-02-22 14:36:58.044440-05:00
2,1.0,1.0,65,232,2021-01-02 00:09:02,2021-01-02 00:15:59,N,1.0,3.70,12.0,3.25,0.5,0.00,0.0,0.3,16.05,2.0,cash,green,2026-02-22 14:36:58.044440-05:00
3,1.0,1.0,75,20,2021-01-02 00:48:21,2021-01-02 01:06:31,N,2.0,9.40,26.0,0.50,0.5,0.02,0.0,0.3,27.32,1.0,credit_card,green,2026-02-22 14:36:58.044440-05:00
4,2.0,1.0,74,41,2021-01-02 00:56:04,2021-01-02 00:57:18,N,1.0,0.37,3.0,0.50,0.5,1.00,0.0,0.3,5.30,1.0,credit_card,green,2026-02-22 14:36:58.044440-05:00


In [31]:
con.sql("SELECT * FROM reports.trips_report").df()

,service_type,pickup_date,total_trips,total_amount,total_fare,total_tips,avg_trip_distance,avg_passenger_count
0,yellow,2021-01-29,50513,815644.01,536520.28,97996.48,2.347302,1.387960
1,yellow,2021-01-30,36838,595523.41,401658.00,71477.92,2.594110,1.444867
2,yellow,2021-01-27,48522,780191.12,514604.20,92474.91,2.340444,1.364494
3,green,2021-01-10,928,15069.04,12182.37,1305.33,3.093513,1.191810
4,green,2021-01-12,1504,27303.82,22047.60,2195.06,3.318118,1.203457
5,green,2021-01-24,879,14873.49,12050.59,1183.40,3.416985,1.187713
6,green,2021-01-26,1301,23618.36,19120.25,1708.89,3.492629,1.109147
7,yellow,2021-01-31,29070,481105.81,326324.76,58397.40,6.699147,1.433127
8,yellow,2021-01-08,46271,753504.07,499559.45,87847.25,2.478124,1.403903
9,yellow,2021-01-16,36281,602829.93,409722.10,71769.28,2.757917,1.463686


In [32]:
con.sql("SHOW ALL TABLES")

┌───────────────┬─────────┬──────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│   database    │ schema  │           name           │                                                                                                                                              

In [33]:
df = pd.DataFrame(con.sql("SHOW ALL TABLES").fetchall(), columns=["database", "schema", "name", "column_names", "column_types", "temporary"])
df

,database,schema,name,column_names,column_types,temporary
0,taxi_rides_ny,dev,dim_vendors,"[vendor_id, vendor_name]","[INTEGER, VARCHAR]",False
1,taxi_rides_ny,dev,dim_zones,"[location_id, borough, zone, service_zone]","[INTEGER, VARCHAR, VARCHAR, VARCHAR]",False
2,taxi_rides_ny,dev,fct_monthly_zone_revenue,"[pickup_zone, revenue_month, service_type, rev...","[VARCHAR, DATE, VARCHAR, DECIMAL(38,3), DECIMA...",False
3,taxi_rides_ny,dev,fct_trips,"[trip_id, vendor_id, service_type, rate_code_i...","[VARCHAR, INTEGER, VARCHAR, INTEGER, INTEGER, ...",False
4,taxi_rides_ny,dev,int_trips,"[trip_id, vendor_id, service_type, rate_code_i...","[VARCHAR, INTEGER, VARCHAR, INTEGER, INTEGER, ...",False
5,taxi_rides_ny,dev,int_trips_unioned,"[vendor_id, rate_code_id, pickup_location_id, ...","[INTEGER, INTEGER, INTEGER, INTEGER, TIMESTAMP...",False
6,taxi_rides_ny,dev,payment_type_lookup,"[payment_type, description]","[INTEGER, VARCHAR]",False
7,taxi_rides_ny,dev,stg_green_tripdata,"[vendor_id, rate_code_id, pickup_location_id, ...","[INTEGER, INTEGER, INTEGER, INTEGER, TIMESTAMP...",False
8,taxi_rides_ny,dev,stg_yellow_tripdata,"[vendor_id, rate_code_id, pickup_location_id, ...","[INTEGER, INTEGER, INTEGER, INTEGER, TIMESTAMP...",False
9,taxi_rides_ny,dev,taxi_zone_lookup,"[locationid, borough, zone, service_zone]","[INTEGER, VARCHAR, VARCHAR, VARCHAR]",False


In [34]:
# con.close()